# Kybalion-1B DPO 학습 & 비교

**Colab 단일 노트북** — `devwoo/Kybalion-1B`을 DPO로 정렬하고, 학습 전/후 답변을 10개 프롬프트로 비교합니다.

- **논문**: Nathan Lambert, *RLHF 책* (arXiv:2504.12501) — §6 직접 정렬 알고리즘
- **알고리즘**: DPO (Rafailov et al. 2023). Zephyr 레시피 (Tunstall et al. 2023) 사용.
- **베이스 모델**: [devwoo/Kybalion-1B](https://huggingface.co/devwoo/Kybalion-1B) — Llama 3.2 1B + CPT + SFT(LoRA)
- **데이터셋**: `argilla/ultrafeedback-binarized-preferences-cleaned` (~62K pair)
- **하드웨어**: A100 40GB (Colab Pro/Pro+). 풀 1 epoch 예상 시간 **3~5시간**

## 사용 순서

1. Colab에서 이 노트북 열고 **런타임 → 런타임 유형 변경 → A100 GPU** 선택
2. 위에서부터 셀 차례로 실행
3. 학습 체크포인트는 `/content/drive/MyDrive/Kybalion-DPO/`에 자동 저장 (끊겨도 재개됨)
4. 학습 끝나면 비교 섹션에서 base vs DPO 응답이 10개 프롬프트에 대해 나옴
5. 만족스러우면 마지막 섹션에서 HuggingFace에 instruct 모델로 push (수동)

## 1. 의존성 설치

버전 호환 검증된 조합 (TRL 0.11.x 계열). 약 2분 소요.

In [ ]:
!pip install -q -U \
    "trl>=0.11.0,<0.13.0" \
    "peft>=0.13.0" \
    "transformers>=4.45.0" \
    "accelerate>=1.0.0" \
    "datasets>=3.0.0" \
    "bitsandbytes>=0.44.0" \
    "torchao>=0.16.0" \
    "sentencepiece" \
    "protobuf"

# torchao 가 Colab 에 0.10.0 으로 미리 설치되어 있어 peft 가 거부함 → 0.16+ 강제 업그레이드
# 설치 후 런타임 재시작 불필요 (import 한 적 없는 라이브러리)

## 2. Google Drive 마운트

체크포인트 + 최종 adapter + 비교 CSV가 Drive에 저장됩니다. Colab 세션 끊겨도 안 잃어버려요.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
SAVE_DIR = '/content/drive/MyDrive/Kybalion-DPO'
os.makedirs(SAVE_DIR, exist_ok=True)
print(f"저장 디렉터리: {SAVE_DIR}")

## 3. 하이퍼파라미터 설정

전부 한 셀에 모아둠. **Zephyr 레시피** (RLHF 책 §6 → §8).

> ⚠️ `LEARNING_RATE=5e-7`은 Zephyr full FT 기준 값. LoRA에는 보수적이라 수렴 느릴 수 있음.
> 학습 중 `rewards/margins`가 500 step 이후에도 0 근처면 `5e-6`으로 올리세요.

> 💡 본 학습 전에 코드 검증부터 하고 싶으면 `MAX_TRAIN_SAMPLES = 1000`으로 (약 10분).

In [ ]:
# ---------------- 모델 / 데이터 ----------------
BASE_MODEL   = "devwoo/Kybalion-1B"
DATASET_NAME = "argilla/ultrafeedback-binarized-preferences-cleaned"
MAX_TRAIN_SAMPLES = None     # None = 전체 ~62K, 스모크는 1000

# ---------------- DPO 하이퍼파라미터 (Zephyr) ----------------
LEARNING_RATE  = 5e-7
BETA           = 0.1
PER_DEVICE_BS  = 8
GRAD_ACCUM     = 4           # effective batch = 32
MAX_LENGTH     = 1024
MAX_PROMPT_LEN = 512
NUM_EPOCHS     = 1
WARMUP_RATIO   = 0.10
LR_SCHEDULER   = "cosine"

# ---------------- LoRA ----------------
LORA_R         = 16
LORA_ALPHA     = 32
LORA_DROPOUT   = 0.05
LORA_TARGETS   = ["q_proj", "k_proj", "v_proj", "o_proj",
                  "gate_proj", "up_proj", "down_proj"]

# ---------------- 로깅 / 체크포인트 ----------------
LOG_STEPS      = 25
SAVE_STEPS     = 200
EVAL_STEPS     = 200
SEED           = 42

# 경로
CHECKPOINT_DIR = f"{SAVE_DIR}/checkpoints"
ADAPTER_PATH   = f"{SAVE_DIR}/adapter-final"
COMPARISON_CSV = f"{SAVE_DIR}/comparison.csv"
COMPARISON_MD  = f"{SAVE_DIR}/comparison.md"

print(f"Effective batch size: {PER_DEVICE_BS * GRAD_ACCUM}")
print(f"LR: {LEARNING_RATE} | beta: {BETA} | epochs: {NUM_EPOCHS}")

## 4. Kybalion-1B + 토크나이저 로드

BF16, 양자화 없이 (1B 정도면 충분히 들어감).

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, PreTrainedTokenizerFast

# Kybalion-1B 의 tokenizer_config.json 에 "tokenizer_class": "TokenizersBackend"
# 라는 비표준 값이 들어있어 AutoTokenizer 가 실패함.
# → AutoTokenizer 우선 시도, 실패 시 PreTrainedTokenizerFast 로 강제 로드.
try:
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
except ValueError as e:
    print(f"AutoTokenizer 실패: {e}")
    print("→ PreTrainedTokenizerFast 로 fallback 로드")
    tokenizer = PreTrainedTokenizerFast.from_pretrained(BASE_MODEL)

# Fallback 로드 시 chat_template 이 비어있는 경우 Llama 3.2 표준 템플릿 주입
# (Kybalion-1B 가 Llama 3.2 기반이므로 동일 템플릿 사용)
if tokenizer.chat_template is None:
    print("→ chat_template 없음, Llama 3.2 표준 템플릿 주입")
    tokenizer.chat_template = (
        "{% set loop_messages = messages %}"
        "{% for message in loop_messages %}"
        "{% set content = '<|start_header_id|>' + message['role'] + '<|end_header_id|>\n\n'"
        " + message['content'] | trim + '<|eot_id|>' %}"
        "{% if loop.index0 == 0 %}{% set content = bos_token + content %}{% endif %}"
        "{{ content }}"
        "{% endfor %}"
        "{% if add_generation_prompt %}"
        "{{ '<|start_header_id|>assistant<|end_header_id|>\n\n' }}"
        "{% endif %}"
    )

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    attn_implementation="sdpa",
)
model.config.use_cache = False  # gradient checkpointing 쓰려면 끄기

print(f"모델: {sum(p.numel() for p in model.parameters()) / 1e6:.0f}M params")
print(f"Vocab: {len(tokenizer)} | EOS: {tokenizer.eos_token} | PAD: {tokenizer.pad_token}")
print(f"Chat template 존재: {tokenizer.chat_template is not None}")

## 5. UltraFeedback 로드 & 포맷팅

데이터셋에서 필요한 필드 3개:
- `prompt`: 유저 질문 (str)
- `chosen`: 선호되는 응답 (메시지 리스트)
- `rejected`: 비선호 응답 (메시지 리스트)

각 row를 Llama 3.2 chat template 적용해서 DPO 학습 포맷 (prompt/chosen/rejected 텍스트 3개)로 변환.

In [ ]:
from datasets import load_dataset

raw = load_dataset(DATASET_NAME)
print(f"Splits: {list(raw.keys())}")
print(f"컬럼: {raw['train'].column_names}")

# 이 데이터셋은 train 스플릿만 제공 → train 에서 200개 떼서 eval 로 사용
EVAL_SIZE = 200
split = raw["train"].train_test_split(test_size=EVAL_SIZE, seed=SEED)
train_raw = split["train"]
eval_raw  = split["test"]

print(f"\nTrain: {len(train_raw):,} | Eval: {len(eval_raw):,}")
print(f"\n첫 번째 chosen[-1]: {train_raw[0]['chosen'][-1]['content'][:200]}...")

In [ ]:
def format_dpo(example):
    """UltraFeedback row → TRL DPO 포맷 (prompt/chosen/rejected 모두 string)."""
    prompt_text = tokenizer.apply_chat_template(
        [{"role": "user", "content": example["prompt"]}],
        tokenize=False,
        add_generation_prompt=True,
    )
    chosen_text   = example["chosen"][-1]["content"]
    rejected_text = example["rejected"][-1]["content"]
    return {"prompt": prompt_text, "chosen": chosen_text, "rejected": rejected_text}


train_ds = train_raw.map(format_dpo, remove_columns=train_raw.column_names)
eval_ds  = eval_raw.map(format_dpo, remove_columns=eval_raw.column_names)

if MAX_TRAIN_SAMPLES is not None:
    train_ds = train_ds.shuffle(seed=SEED).select(range(min(MAX_TRAIN_SAMPLES, len(train_ds))))
    print(f"⚠️  스모크 모드: {len(train_ds):,} 샘플만 사용")

print(f"최종 train: {len(train_ds):,} | eval: {len(eval_ds):,}")
print(f"\n샘플 prompt:\n{train_ds[0]['prompt'][:300]}...")
print(f"\n샘플 chosen:   {train_ds[0]['chosen'][:120]}...")
print(f"샘플 rejected: {train_ds[0]['rejected'][:120]}...")

## 6. DPO Trainer 셋업

- `LoraConfig` — r=16, 모든 linear 레이어
- `DPOConfig` — Zephyr 하이퍼파라미터
- Reference 모델은 TRL+PEFT가 자동 처리 (LoRA adapter toggle, 추가 메모리 0)

In [ ]:
from peft import LoraConfig
from trl import DPOConfig, DPOTrainer

peft_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=LORA_TARGETS,
    bias="none",
    task_type="CAUSAL_LM",
)

training_args = DPOConfig(
    output_dir=CHECKPOINT_DIR,
    per_device_train_batch_size=PER_DEVICE_BS,
    per_device_eval_batch_size=PER_DEVICE_BS,
    gradient_accumulation_steps=GRAD_ACCUM,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type=LR_SCHEDULER,
    warmup_ratio=WARMUP_RATIO,
    bf16=True,
    tf32=True,
    logging_steps=LOG_STEPS,
    save_steps=SAVE_STEPS,
    save_total_limit=3,
    eval_strategy="steps",
    eval_steps=EVAL_STEPS,
    beta=BETA,
    max_length=MAX_LENGTH,
    max_prompt_length=MAX_PROMPT_LEN,
    loss_type="sigmoid",
    seed=SEED,
    report_to="none",            # wandb 쓰려면 "wandb"
    remove_unused_columns=False,
    optim="adamw_torch",
    dataloader_num_workers=2,
)

trainer = DPOTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    tokenizer=tokenizer,
    peft_config=peft_config,
)

# Sanity check: 학습 가능한 파라미터 수 출력 (LoRA만)
trainer.model.print_trainable_parameters()

## 7. 학습

A100 40GB에서 풀 1 epoch (~62K pair) ≈ **3~5시간**.

**세션 끊기면** 이 셀 그냥 다시 실행. `CHECKPOINT_DIR`에서 최신 체크포인트 자동 감지해서 이어 학습.

In [ ]:
# CHECKPOINT_DIR에 기존 체크포인트 있으면 resume
import os
resume = None
if os.path.isdir(CHECKPOINT_DIR):
    ckpts = [d for d in os.listdir(CHECKPOINT_DIR) if d.startswith("checkpoint-")]
    if ckpts:
        resume = True
        print(f"기존 체크포인트 {len(ckpts)}개 발견 — resume합니다.")

trainer.train(resume_from_checkpoint=resume)

## 8. 최종 adapter를 Drive에 저장

중간 체크포인트와 별도로 최종 LoRA adapter + 토크나이저만 따로 보관.

In [ ]:
trainer.save_model(ADAPTER_PATH)
tokenizer.save_pretrained(ADAPTER_PATH)
print(f"✅ 최종 adapter 저장: {ADAPTER_PATH}")
print(f"   크기: {sum(os.path.getsize(os.path.join(ADAPTER_PATH, f)) for f in os.listdir(ADAPTER_PATH)) / 1e6:.1f} MB")

## 9. Base vs DPO-trained 비교

10개 프롬프트 — helpfulness / reasoning / coding / advice / creative 골고루.

같은 sampling 설정으로 각 프롬프트를 **두 번** 생성:
- **Base**: `model.disable_adapter()` (LoRA off → 원본 Kybalion-1B)
- **Trained**: 기본 상태 (LoRA on → DPO 정렬 모델)

같은 모델 객체로 toggle만 하니까 추가 메모리 ~0.

In [ ]:
TEST_PROMPTS = [
    "Explain quantum entanglement in simple terms a high-schooler could grasp.",
    "If a train leaves city A at 3 PM going 60 mph east, and another leaves city B (200 miles east of A) at 4 PM going 80 mph west, when do they meet?",
    "How can I improve my home security on a tight budget under $200?",
    "Write a haiku about debugging code at 2 AM.",
    "List 5 productivity tips for a remote worker, each in exactly one sentence.",
    "Write a Python function `is_palindrome(s)` that returns True if `s` is a palindrome, ignoring case and non-alphanumeric characters.",
    "I'm feeling overwhelmed with work and not sleeping well. What concrete steps should I take this week?",
    "Why is the sky blue? Explain at a 6th-grader level.",
    "Plan a 3-day trip to Tokyo for two first-time visitors arriving on a Friday morning.",
    "What are 3 pros and 3 cons of remote work, with one concrete example each?",
]
GEN_KW = dict(max_new_tokens=400, temperature=0.7, top_p=0.9, do_sample=True, repetition_penalty=1.05)
print(f"총 {len(TEST_PROMPTS) * 2}개 응답 생성 예정.")

In [ ]:
# PEFT 모델 확보 — 두 경로 자동 분기:
#   (A) 방금 이 세션에서 학습 완료 → trainer.model 사용
#   (B) 학습 건너뛰고 저장된 adapter 만 로드 → PeftModel.from_pretrained
try:
    peft_model = trainer.model
    print("✓ trainer.model 사용 (이번 세션에서 학습됨)")
except NameError:
    from peft import PeftModel
    print(f"✓ 저장된 adapter 로드: {ADAPTER_PATH}")
    peft_model = PeftModel.from_pretrained(model, ADAPTER_PATH)

peft_model.eval()


@torch.no_grad()
def generate(prompt: str) -> str:
    messages = [{"role": "user", "content": prompt}]
    input_text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(input_text, return_tensors="pt").to(peft_model.device)
    out = peft_model.generate(
        **inputs, pad_token_id=tokenizer.eos_token_id, **GEN_KW
    )
    gen = out[0][inputs.input_ids.shape[1]:]
    return tokenizer.decode(gen, skip_special_tokens=True).strip()


from tqdm.auto import tqdm

print("\n--- BASE 생성 (LoRA OFF) ---")
base_responses = []
with peft_model.disable_adapter():
    for p in tqdm(TEST_PROMPTS):
        base_responses.append(generate(p))

print("\n--- DPO-TRAINED 생성 (LoRA ON) ---")
trained_responses = []
for p in tqdm(TEST_PROMPTS):
    trained_responses.append(generate(p))

In [ ]:
import pandas as pd
from IPython.display import HTML, display

rows = []
for i, (p, b, t) in enumerate(zip(TEST_PROMPTS, base_responses, trained_responses)):
    rows.append({"#": i + 1, "Prompt": p, "Base (Kybalion-1B)": b, "DPO Trained": t})

df = pd.DataFrame(rows)

# Drive에 저장
df.to_csv(COMPARISON_CSV, index=False)
df.to_markdown(COMPARISON_MD, index=False)
print(f"✅ 비교 저장:\n   {COMPARISON_CSV}\n   {COMPARISON_MD}")

# 노트북 안에 side-by-side 렌더
pd.set_option('display.max_colwidth', None)
html = df.to_html(index=False, escape=True).replace(
    '<table',
    '<table style="border-collapse:collapse;font-size:13px;table-layout:fixed;width:100%"'
)
html = html.replace('<th>Base (Kybalion-1B)</th>',
                    '<th style="width:42%;text-align:left">Base (Kybalion-1B)</th>')
html = html.replace('<th>DPO Trained</th>',
                    '<th style="width:42%;text-align:left">DPO Trained</th>')
html = html.replace('<td>', '<td style="vertical-align:top;padding:8px;border:1px solid #ddd;white-space:pre-wrap">')
display(HTML(html))

## 10. 한 프롬프트씩 풀텍스트로 보기

표가 좁아서 보기 힘들면 이 셀로 프롬프트별 전체 응답을 출력.

In [ ]:
for i, (p, b, t) in enumerate(zip(TEST_PROMPTS, base_responses, trained_responses)):
    print("=" * 90)
    print(f"PROMPT {i+1}: {p}")
    print("=" * 90)
    print("\n[Base (Kybalion-1B)]")
    print(b)
    print("\n[DPO Trained]")
    print(t)
    print()

## 11. (선택) HuggingFace에 instruct 모델로 push

> ⚠️ 비교 결과 확인하고 **마음에 들 때만** 수동으로 실행하세요.
> 아래 셀들은 LoRA adapter를 base에 merge해서 standalone 모델로 만들어 push합니다.

순서:
1. `notebook_login()` 한 번 실행해 HF 인증
2. 타겟 repo 이름 설정
3. LoRA를 base에 merge
4. merged 모델 + 토크나이저 push

LoRA adapter만 Drive에 보관하고 끝낼 거면 이 섹션 건너뛰셔도 됩니다.

In [ ]:
# 세션당 1회만 실행
# from huggingface_hub import notebook_login
# notebook_login()

In [ ]:
# 타겟 repo. 원하는 이름으로 바꾸세요.
HF_REPO = "devwoo/Kybalion-1B-DPO"   # 또는 "devwoo/Kybalion-1B-Instruct"
PRIVATE = True                         # False로 바꾸면 public

In [ ]:
# LoRA를 base에 merge하고 local에 merged 모델 저장
from peft import PeftModel
from transformers import AutoModelForCausalLM

print("Base를 BF16으로 다시 로드...")
base_for_merge = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
print("Adapter 로드 후 merge...")
merged = PeftModel.from_pretrained(base_for_merge, ADAPTER_PATH)
merged = merged.merge_and_unload()

MERGED_DIR = f"{SAVE_DIR}/merged"
merged.save_pretrained(MERGED_DIR, safe_serialization=True)
tokenizer.save_pretrained(MERGED_DIR)
print(f"✅ Merged 모델: {MERGED_DIR}")

In [ ]:
# Merged 모델 + 토크나이저 push
# from huggingface_hub import HfApi
# api = HfApi()
# api.create_repo(HF_REPO, private=PRIVATE, exist_ok=True)
# merged.push_to_hub(HF_REPO, private=PRIVATE)
# tokenizer.push_to_hub(HF_REPO, private=PRIVATE)
# print(f"✅ Pushed: https://huggingface.co/{HF_REPO}")

# 위 코드는 주석 처리되어 있습니다. 실제로 push할 때만 주석 풀고 실행하세요.
# (실수로 공개 안 되도록 안전장치)

## 부록 — 레퍼런스 레시피와의 차이점

Zephyr DPO 레시피 (RLHF 책 §6 → §8)는 Mistral-7B + full fine-tuning 기준. 이 노트북은 아래처럼 다릅니다:

| # | 항목 | Zephyr | 본 구현 | 이유 |
|---|------|--------|--------|------|
| 1 | 베이스 모델 | Mistral-7B | **Llama 3.2 1B (Kybalion)** | 사용자 모델 |
| 2 | 어댑터 | Full FT | **LoRA r=16, all linear** | A100 40GB + storage + 재현성 |
| 3 | Reference 모델 | 별도 frozen copy | TRL+PEFT adapter toggle | 수학 동등, ~2.5 GB 절약 |
| 4 | Max length | 2048 | 1024 | 시간/메모리 trade-off |
| 5 | Learning rate | 5e-7 (full FT용) | 5e-7 with LoRA ⚠️ | 보수적 — `rewards/margins`가 500 step 후에도 0 근처면 5e-6으로 올리세요 |
| 6 | Chat template | Mistral | Llama 3.2 | 모델 선택에 따른 자연스러운 변경 |

알고리즘 레벨 (`loss_type=sigmoid`, β=0.1, batch=32, 1 epoch, cosine 10% warmup, UltraFeedback 데이터셋)은 **Zephyr 레시피 그대로**.

더 자세한 deviation 분석은 같은 저자의 Agentic_Autosurvey 프로젝트의 deviation audit 섹션을 참고하세요.